# From Design to Foundry — PDK and DRC Flow

Photonic integrated circuits are fabricated at specialised foundries that provide a
**Process Design Kit (PDK)** defining the available layer stack, component models, and
**design rules** (minimum widths, spacings, bend radii). Before submitting a design for
fabrication ("tapeout"), you must pass **Design Rule Checking (DRC)** to ensure the
layout is manufacturable.

This notebook shows how to run the full design-to-DRC flow using `design_pic`.

In [ ]:
from photonstrust.easy import design_pic

In [ ]:
result = design_pic(
    components=["mmi_2x2", "mmi_2x2"],
    connections=[{"from": [0, "out1"], "to": [1, "in1"]}],
    pdk="generic_silicon_photonics",
    wavelength_nm=1550.0,
    run_drc=True,
)
print(result.summary())

## Understanding DRC Results

The summary indicates whether the design passed all foundry rules. Common violations
include minimum waveguide width, insufficient spacing between components, and bend
radius violations. You can inspect the full DRC report from the result dictionary.

In [ ]:
import json
details = result.as_dict()
drc_section = {k: v for k, v in details.items() if "drc" in k.lower()}
print(json.dumps(drc_section if drc_section else details, indent=2, default=str))

## Available PDKs

PhotonsTrust ships with PDK configuration files in `configs/pdks/`. Each file
describes a foundry process with its layer stack, component library, and design rules.

In [ ]:
from pathlib import Path

pdk_dir = Path("configs/pdks")
if pdk_dir.exists():
    pdks = sorted(pdk_dir.glob("*.pdk.json"))
    print(f"Found {len(pdks)} PDK(s):")
    for p in pdks:
        print(f"  {p.stem}")
else:
    print("PDK directory not found — check your installation.")

## Try It Yourself

- Try a different PDK (e.g., `"sin_photonics"` if available) and compare the design
  rules with silicon photonics.
- Intentionally violate a design rule (e.g., very short waveguide spacing) and observe
  the DRC error messages.
- Add more components (ring resonator, grating coupler) and check if they pass DRC.